In [74]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from pydantic import BaseModel, ValidationError
from dotenv import load_dotenv

In [75]:
load_dotenv()

True

In [78]:
class VisitSummary(BaseModel):
    patient: str                              # required
    visit_date: str | None = None             # may be empty
    main_complaint: str                       # required
    diagnosis: list[str] = []                 # may be empty
    medication_name: str | None = None        # may be empty
    medication_dose: str | None = None        # may be empty
    medication_frequency: str | None = None   # may be empty
    plan: list[str] = []                      # may be empty
    follow_up: str | None = None              # may be empty

In [79]:
model=ChatGoogleGenerativeAI(
    model='gemini-3.5-flash',
    temperature=0.2,
    response_mime_type="application/json",
    #max_tokens=300
    timeout=30,
    max_retries=1
    )

In [80]:
structured_model = model.with_structured_output(VisitSummary)

In [69]:
note="Note 1: Routine Adult Consultation (Cardiovascular & Lifestyle) Patient: Arthur Pendelton, 58M Date of Visit: October 8, 2026 Chief Complaint: Annual comprehensive exam and follow-up on essential hypertension. Subjective: Patient reports feeling generally well with mild fatigue during afternoon work hours. Denies chest pain, shortness of breath, palpitations, orthopnea, or lower extremity edema. Compliance with prescribed daily Lisinopril 20mg is reported as good, though he admits to skipping weekend doses occasionally. Diet consists heavily of processed foods due to a busy work routine, and alcohol intake is reported at 3 to 4 units on weekends. Physical activity remains limited to weekly casual walking. Objective: Vital Signs: BP 138/86 mmHg, HR 72 bpm (regular), RR 16 bpm, SpO2 98% on room air, BMI 29.4 kg/m². Physical Exam: HEENT intact. Cardiovascular: Normal S1/S2, no murmurs, gallops, or rubs. Lungs clear to auscultation bilaterally. Abdomen soft, non-tender, no organomegaly. No peripheral edema noted. Lab results from prior week indicate total cholesterol of 215 mg/dL, LDL 135 mg/dL, HDL 42 mg/dL, Triglycerides 190 mg/dL, and HbA1c of 5.7%. Assessment & Plan:Essential Hypertension: Suboptimally controlled. Strongly emphasized the importance of strict medication adherence without missing weekend doses. Hyperlipidemia: Elevated LDL and triglycerides. Recommended dietary modification with reduced intake of saturated fats and sodium. Patient agreed to consult with a registered dietitian. Preventive Care: Encouraged increasing aerobic exercise to 150 minutes per week. Re-check BP and routine labs in 3 months."

In [70]:
system_prompt = """You extract key information from visit notes.
Return only valid JSON with exactly these keys:
{
  "patient": "name, age and sex",
  "visit_date": "",
  "main_complaint": "",
  "symptoms": [],
  "vitals": {},
  "lab_results": {},
  "diagnosis": [],
  "medications": [],
  "lifestyle_factors": [],
  "plan": [],
  "follow_up": ""
}
Rules:
- Use only facts stated in the note.
- If something is not mentioned, use null (or [] for lists, {} for objects).
- Keep values short, no long paragraphs."""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=note),   # your visit note text
]

In [81]:
response = model.invoke(messages)

print(response.content)
print(response.usage_metadata)

[{'type': 'text', 'text': '{\n  "patient": "Arthur Pendelton, 58-year-old male",\n  "visit_date": "October 8, 2026",\n  "main_complaint": "Annual comprehensive exam and follow-up on essential hypertension",\n  "symptoms": [\n    "Mild fatigue"\n  ],\n  "vitals": {\n    "BP": "138/86 mmHg",\n    "HR": "72 bpm",\n    "RR": "16 bpm",\n    "SpO2": "98% on room air",\n    "BMI": "29.4 kg/m²"\n  },\n  "lab_results": {\n    "Total Cholesterol": "215 mg/dL",\n    "LDL": "135 mg/dL",\n    "HDL": "42 mg/dL",\n    "Triglycerides": "190 mg/dL",\n    "HbA1c": "5.7%"\n  },\n  "diagnosis": [\n    "Essential Hypertension",\n    "Hyperlipidemia"\n  ],\n  "medications": [\n    "Lisinopril 20mg daily"\n  ],\n  "lifestyle_factors": [\n    "Diet heavy in processed foods",\n    "Alcohol intake of 3 to 4 units on weekends",\n    "Physical activity limited to weekly casual walking"\n  ],\n  "plan": [\n    "Emphasized strict medication adherence (no skipping weekend doses)",\n    "Dietary modification with red

In [82]:
print(response.content)

[{'type': 'text', 'text': '{\n  "patient": "Arthur Pendelton, 58-year-old male",\n  "visit_date": "October 8, 2026",\n  "main_complaint": "Annual comprehensive exam and follow-up on essential hypertension",\n  "symptoms": [\n    "Mild fatigue"\n  ],\n  "vitals": {\n    "BP": "138/86 mmHg",\n    "HR": "72 bpm",\n    "RR": "16 bpm",\n    "SpO2": "98% on room air",\n    "BMI": "29.4 kg/m²"\n  },\n  "lab_results": {\n    "Total Cholesterol": "215 mg/dL",\n    "LDL": "135 mg/dL",\n    "HDL": "42 mg/dL",\n    "Triglycerides": "190 mg/dL",\n    "HbA1c": "5.7%"\n  },\n  "diagnosis": [\n    "Essential Hypertension",\n    "Hyperlipidemia"\n  ],\n  "medications": [\n    "Lisinopril 20mg daily"\n  ],\n  "lifestyle_factors": [\n    "Diet heavy in processed foods",\n    "Alcohol intake of 3 to 4 units on weekends",\n    "Physical activity limited to weekly casual walking"\n  ],\n  "plan": [\n    "Emphasized strict medication adherence (no skipping weekend doses)",\n    "Dietary modification with red